In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'jax.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Implementation

In that section we chose the weights of a hidden layer by hand, which
is feasible for the four points of XOR and for little else. Let us now leave
the choice to training, on real images. On Fashion-MNIST, softmax
regression reached about 83% validation accuracy after ten epochs, and most
of its errors confused garments of similar outline, such as shirts,
pullovers, and coats (that section). Telling such garments
apart depends on how pixels combine, which is what a hidden layer can
compute. How many of the errors does one hidden layer remove, which ones,
and what does it cost? On paper the cost is considerable: with 256 hidden
units the MLP
has 26 times the parameters of softmax regression (that section) and
performs 26 times the arithmetic on every image. Whether the gain in
accuracy justifies this extra work is a question for experiment.

We implement the MLP twice and train it as we trained softmax regression.
The first implementation writes the four parameter arrays, the activation
function, and the forward pass with tensor operations, and it checks the
untrained model before any training time is spent. The second assembles the
same model from the framework's layers. The data, the cross-entropy loss,
minibatch stochastic gradient descent (SGD), and the `Trainer` stay as they
were, so any change in accuracy comes from the hidden layer. Finally we
compare the MLP with softmax regression trained under the same settings, in
accuracy, class by class, and in training time, and we look at what its
hidden units compute. Automatic differentiation computes
all gradients in this section; the next section shows how.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

In [ ]:
import time

## Implementation from Scratch

### The Parameters

Each $28 \times 28$ image becomes a vector of $d = 784$ pixel values, and
there are $q = 10$ classes. We give the hidden layer $h = 256$ units
(the figure). The width is a hyperparameter, like the
learning rate: training does not choose it, and Exercise
4 compares other widths. In the notation
of that section, each layer has a weight matrix with one row per input
and one column per output, and a bias vector with one entry per output:

$$\mathbf{W}^{(1)} \in \mathbb{R}^{784 \times 256}, \quad \mathbf{b}^{(1)} \in \mathbb{R}^{256}, \quad \mathbf{W}^{(2)} \in \mathbb{R}^{256 \times 10}, \quad \mathbf{b}^{(2)} \in \mathbb{R}^{10}.$$

Together these hold the 203,530 numbers counted in that section. All
but 2,570 of them belong to the hidden layer.

![The MLP of this section. A minibatch of $n$ images is flattened to an $n \times 784$ matrix $\mathbf{X}$, an affine map followed by the ReLU computes the $n \times 256$ hidden representation $\mathbf{H}$, and a second affine map computes the $n \times 10$ logits $\mathbf{O}$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-mlp-arch.svg)

How should the four arrays start? As for softmax regression, we draw the
weights from a normal distribution with mean 0 and standard deviation 0.01
and set the biases to 0. The weights must be random: hidden units that
start with equal weights receive equal gradients and stay identical
throughout training, as that section shows. Small
weights keep the logits close to 0 at first, so the untrained model gives
every class a probability close to 1/10. The scale 0.01 itself is merely
inherited from softmax regression, which keeps the comparison between the
two models clean. It is not a good scale for a network with a hidden layer,
and the first epochs of training will show it.

We register the four arrays as parameters of the module, as in
that section. The default optimizer of `Classifier`,
minibatch SGD at the learning rate `lr`, then finds them by itself, and the
model needs no `configure_optimizers` of its own. The argument `std` sets
the standard deviation of the initial weights.

As in that section, the argument `rngs` supplies the random
key for the initialization.

In [ ]:
class MLPScratch(d2l.Classifier):
    """An MLP with one hidden layer, implemented from scratch."""
    def __init__(self, num_inputs, num_outputs, num_hiddens, lr, std=0.01,
                 rngs=None):
        super().__init__()
        self.save_hyperparameters(ignore=['rngs'])
        rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
        self.W1 = nnx.Param(
            rngs.params.normal((num_inputs, num_hiddens)) * std)
        self.b1 = nnx.Param(jnp.zeros(num_hiddens))
        self.W2 = nnx.Param(
            rngs.params.normal((num_hiddens, num_outputs)) * std)
        self.b2 = nnx.Param(jnp.zeros(num_outputs))

### The Forward Pass

Let us start with the activation function, which we write ourselves. The
ReLU keeps the positive entries and zeros the others, so it multiplies each
entry $z$ by 1 if $z > 0$ and by 0 otherwise. (Written as a product with
this mask, it also gets the derivative 0 at $z = 0$ from automatic
differentiation, as the built-in ReLU does. A maximum with 0 may get 1 or
$1/2$, depending on the framework, and
that section explained why we need not care.)

In [ ]:
def relu(X):
    return X * (X > 0)

For a minibatch of $n$ images, flattened into the rows of
$\mathbf{X} \in \mathbb{R}^{n \times 784}$, the model computes

$$\mathbf{H} = \mathrm{ReLU}\left(\mathbf{X}\mathbf{W}^{(1)} + \mathbf{b}^{(1)}\right), \qquad \mathbf{O} = \mathbf{H}\mathbf{W}^{(2)} + \mathbf{b}^{(2)}.$$

Row $i$ of $\mathbf{H} \in \mathbb{R}^{n \times 256}$ is the hidden
representation of image $i$, and row $i$ of
$\mathbf{O} \in \mathbb{R}^{n \times 10}$ holds its logits; the biases are
added to every row by broadcasting. The method `hidden` computes
$\mathbf{H}$, and `forward` computes $\mathbf{O}$ from it. That is the
whole model. Note that it returns logits, not probabilities: the loss that
every `Classifier` inherits computes the cross-entropy from the logits
(that section).

In [ ]:
@d2l.add_to_class(MLPScratch)
def hidden(self, X):
    X = d2l.reshape(X, (-1, self.num_inputs))  # One flattened image per row
    return relu(d2l.matmul(X, self.W1) + self.b1)

@d2l.add_to_class(MLPScratch)
def forward(self, X):
    return d2l.matmul(self.hidden(X), self.W2) + self.b2  # The logits

### Checks before Training

Training takes minutes, and a mistake in the model may only show as poor
accuracy at the end. A forward pass through the untrained model on a few
images tests three predictions in a fraction of a second. First, each image
should have a hidden representation of 256 entries and one logit per class.
Second, since the logits start close to 0, the loss should be close to
$\log 10 \approx 2.30$, the cross-entropy of a prediction that gives every
class the probability 1/10. Third, about half of the hidden activations
should be positive. With zero biases, the pre-activation of a hidden unit is
a sum of pixel values times weights drawn symmetrically around 0, so it is
positive or negative with equal probability, and the ReLU sets the negative
ones to 0.

In [ ]:
data = d2l.FashionMNIST(batch_size=256)
model = MLPScratch(num_inputs=784, num_outputs=10, num_hiddens=256, lr=0.1)
X, y = next(iter(data.train_dataloader()))
X, y = X[:8], y[:8]  # Eight images suffice for the checks
H, O = model.hidden(X), model(X)
active = d2l.reduce_mean(d2l.astype(H > 0, d2l.float32))
print('H:', H.shape, 'O:', O.shape)
print(f'loss {d2l.numpy(model.loss(O, y)):.3f}, '
      f'fraction of active units {d2l.numpy(active):.2f}')

The shapes are $8 \times 256$ and $8 \times 10$, the loss is close to 2.30,
and about half of the hidden units are active. A weight matrix with its axes
swapped would stop the forward pass with an error, a wrong number of inputs
would show up in the shapes, and a loss far from 2.30 would point to weights
that are too large or to a mistake in the loss.

### Training

We train with the minibatch size and the learning rate of softmax regression
in that section, 256 images and 0.1, but for 30 epochs
instead of 10. The longer run is a consequence of our initialization: with
weights this small the hidden layer starts slowly, as the comparison at the
end of this section shows. With `num_gpus=1` the `Trainer` trains on a GPU
if the machine has one and on the CPU otherwise (that section).

In [ ]:
trainer = d2l.Trainer(max_epochs=30, num_gpus=1)
trainer.fit(model, data)

The training loss falls over all 30 epochs. After the first few epochs the
validation loss lies above it and falls more slowly, and the validation
accuracy climbs for most of the run. The climb is far from smooth. From one
epoch to the next the accuracy moves by about a point, and a single epoch
can fall several points below its neighbors. Nothing is broken: minibatch
SGD at a constant learning rate never settles, and we measure the validation
accuracy wherever the last step of the epoch happened to leave the
parameters.

## Concise Implementation

Nobody who builds networks for a living registers four arrays by hand. A
fully connected layer of the framework holds a weight matrix and a bias
vector and computes the affine map of one layer, as in the concise softmax
regression of that section. The concise MLP applies four
operations in order: flattening, a fully connected layer with 256 outputs,
the ReLU, and a fully connected layer with 10 outputs. The framework's
sequential container holds the layers that perform them. We initialize the
model like the scratch model. Left to its default, each framework would
draw the weights from a distribution of its own, and the scratch and the
concise model would start from different scales.

`nnx.relu` enters the sequence as a plain function, each `nnx.Linear` needs
its number of inputs up front, and `forward` flattens the images before it
calls the sequence.

In [ ]:
class MLP(d2l.Classifier):
    """An MLP with one hidden layer, built from framework layers."""
    def __init__(self, num_outputs, num_hiddens, lr, num_inputs=784,
                 rngs=None):
        super().__init__()
        self.save_hyperparameters(ignore=['rngs'])
        rngs = nnx.Rngs(d2l.get_key()) if rngs is None else rngs
        normal = nnx.initializers.normal(0.01)
        self.net = nnx.Sequential(
            nnx.Linear(num_inputs, num_hiddens, kernel_init=normal,
                       rngs=rngs),
            nnx.relu,
            nnx.Linear(num_hiddens, num_outputs, kernel_init=normal,
                       rngs=rngs))

    def forward(self, X):
        return self.net(X.reshape((X.shape[0], -1)))  # Flatten each image

The scratch model spelled out every array and every operation. Here the
layers hold the parameters and the container fixes the order of the
operations, and this is the form that scales: with a deeper network, or one
whose layers we want to reorder, nothing has to be named and registered by
hand.

## Comparison with Softmax Regression

### Accuracy

We now train the concise MLP and, for comparison, the concise softmax
regression of that section, `d2l.SoftmaxRegression`, with
the same trainer: 30 epochs, minibatches of 256, and learning rate 0.1. All
three models start from the same initialization. We time each run and keep
its validation accuracy after every epoch from `trainer.history`. The
trainer's plot is switched off, and we draw the validation curves of the
three models, the scratch MLP included, in one plot instead.

In [ ]:
curves, seconds = {'MLP from scratch': trainer.history['val_acc']}, {}
trainer.board.display = False
models = {'concise MLP': MLP(num_outputs=10, num_hiddens=256, lr=0.1),
          'softmax regression': d2l.SoftmaxRegression(num_outputs=10, lr=0.1)}
for name, net in models.items():
    start = time.perf_counter()
    trainer.fit(net, data)
    seconds[name] = (time.perf_counter() - start) / trainer.max_epochs
    curves[name] = trainer.history['val_acc']

In [ ]:
d2l.plot(list(range(1, trainer.max_epochs + 1)), list(curves.values()),
         'epoch', 'validation accuracy', legend=list(curves))

In [ ]:
print('validation accuracy averaged over the last 5 epochs')
for name, acc in curves.items():  # A single epoch can land a few points low
    print(f'  {name}: {sum(acc[-5:]) / 5:.3f}')
train_acc = trainer.evaluate(model, data, train=True)['acc']
print(f'training accuracy of the MLP from scratch: {train_acc:.3f}')

The scratch and the concise MLP follow the same curve up to the variation
from run to run, as they should: they are the same model, written in two
ways. Both start more slowly than softmax regression and usually trail it
by a few points after the first epoch. Our initialization is to blame.
With weights of standard deviation 0.01, the hidden activations are small,
and every gradient that reaches $\mathbf{W}^{(1)}$ is multiplied by entries
of $\mathbf{W}^{(2)}$, which are just as small, so at first the hidden
layer learns slowly. After ten epochs, the training
length of that section, the MLP is ahead by about a point
on average, within its variation from epoch to epoch. Had we stopped there,
we could hardly have told the two models apart. Since any single epoch can
land a few points low, we compare the models by their validation accuracy
averaged over the last five epochs: softmax regression levels off at about
84%, while the MLP climbs to about 87%. Three points sound modest. Read as
errors, they are more: the hidden layer removes about a fifth of the errors
of softmax regression. On its own training images the MLP is right about
89% of the time, only a few points more often than on the validation
images.

### Errors by Class

Which fifth? It is tempting to assume that it is the shirts taken for
pullovers and coats, the errors we set out to remove. A total accuracy
cannot tell us. We therefore count, for each class, the fraction of its
validation images that a model classifies correctly. This is the *recall* of
the class, the diagonal of the confusion matrix of
that section. The table that follows has a useful feature:
its two MLPs are the same model trained twice, so wherever their columns
disagree, we see how much a single run varies.

In [ ]:
def recall(net):
    """Fraction of the validation images of each class that net gets right."""
    correct, total = np.zeros(10), np.zeros(10)
    for X, y in map(trainer.prepare_batch, data.val_dataloader()):
        y_hat = d2l.numpy(d2l.argmax(net(X), axis=1))
        y = d2l.numpy(y).astype(int)
        total += np.bincount(y, minlength=10)
        correct += np.bincount(y, weights=y_hat == y, minlength=10)
    return correct / total

recalls = {'softmax': recall(models['softmax regression']),
           'MLP, scratch': recall(model),
           'MLP, concise': recall(models['concise MLP'])}
names = data.text_labels(range(10))
print(f'{"class":11s}' + ''.join(f'{name:>14s}' for name in recalls))
for j, name in enumerate(names):
    print(f'{name:11s}' + ''.join(f'{r[j]:14.2f}' for r in recalls.values()))

The answer is less tidy than the question. Start with the six classes that
softmax regression already recognized well: trousers, dresses, bags, and the
three kinds of shoes. Their recall changes by a few points, more often up
than down, and summed over the six, both MLPs make fewer errors than softmax
regression. Among the four tops (T-shirts, pullovers, coats, and shirts)
the table is harder to read, because the two MLPs can disagree with each
other by ten points or more on a single class, although they are the same
model. What we see there is a boundary in motion. The last steps of SGD at a
constant learning rate push the boundaries between similar garments back
and forth, and what one top gains another loses. Only the sum is stable.
Over the four tops together the MLP usually makes fewer errors than softmax
regression, often by a few hundred, which is most of what it saves; after an
unlucky last epoch it can even make more. So we cannot say that the shirts
gain most, and neither does any other single class. The tops remain where
most of the errors are.

### The Learned Features

In that section the weights themselves showed what softmax
regression had learned: one template per class, each a blurred garment. For
XOR we chose the hidden features by hand, and we said that for images
training would have to choose them. What has it chosen? The weights tell
again. Column $j$ of $\mathbf{W}^{(1)}$ holds the weights of the 784 pixels
into hidden unit $j$. Reshaped to $28 \times 28$, it is an image: a bright
pixel where it is red raises the pre-activation of the unit, and a bright
pixel where it is blue lowers it. Not every one of the 256 units makes a
clear picture, so we draw the ten units of the scratch MLP with the largest
incoming weights, each on its own color scale. Each is titled
with the class whose logit it raises most, the largest entry of its row of
$\mathbf{W}^{(2)}$.

In [ ]:
W1, W2 = d2l.numpy(model.W1[...]), d2l.numpy(model.W2[...])

In [ ]:
units = np.argsort(np.linalg.norm(W1, axis=0))[::-1][:10]  # Largest weights
images = W1[:, units].T.reshape(10, 28, 28)
images /= np.abs(images).max(axis=(1, 2), keepdims=True)  # Own color scale
d2l.show_images(images, 2, 5, titles=[names[k] for k in W2[units].argmax(1)],
                cmap='RdBu_r', vmin=-1, vmax=1);

These are not ten more templates. Few of the units look like a whole
garment. Most of them weigh a region: a neckline, the edge of a sleeve, the
gap between the legs of a pair of trousers. Often red lies right beside
blue, so that the unit responds to a contrast between neighboring pixels and
not to brightness alone. A template asks how much an image looks like a
shirt. A hidden unit asks a smaller question, about one part of the image,
and the output layer weighs 256 such answers. Note also the titles: most of
the ten units raise the logit of one of the four tops, the shirt most often.
It is at least suggestive that the units which training moved furthest
serve the classes that softmax regression confused most, though ten units
out of 256 do not prove it.

### Parameters, Computation, and Time

The following table collects what the hidden layer adds. The multiply-adds
are those of the forward pass for one image, $784 \cdot 10 = 7{,}840$ for
softmax regression and $784 \cdot 256 + 256 \cdot 10 = 203{,}264$ for the
MLP.

| model | parameters | multiply-adds per image | validation accuracy, epochs 26–30 |
|:--|--:|--:|--:|
| softmax regression | 7,850 | 7,840 | about 84% |
| MLP, 256 hidden units | 203,530 | 203,264 | about 87% |

The hidden layer multiplies the parameters and the arithmetic by 26. Does
it multiply the training time as well?

In [ ]:
for name, s in seconds.items():
    print(f'{name}: {s:.1f} s per epoch')

Surprisingly, on a GPU it does not: the two times are about the same. For
models this small, nearly all of an epoch goes to reading the images and to
the fixed cost of every step, which do not grow with the width of the
model; Exercise 4 measures how much.

The experiment leaves three questions open, and they are the three problems
of the chapter introduction. Automatic differentiation computed the
gradients of some two hundred thousand parameters without our looking
inside; the next section derives them. Weights of standard deviation 0.01
slowed the start of training, and with more hidden layers they stop it
altogether, as that section will show. And a model with
3.4 parameters per training image could memorize its training set. Ours did
nothing of the kind: it is hardly more accurate on the images it was fitted
to than on those it never saw.

## Summary

A multilayer perceptron with one hidden layer adds a weight matrix, a bias
vector, and an activation function to softmax regression, and everything
else in the training code stays as it was. From scratch, it consists of four
registered parameter arrays, the ReLU written by hand, and a forward
pass of two lines that returns logits. Before training, a forward pass
through the untrained model checks the shapes, the initial loss, and the
fraction of active hidden units against values we can predict. The concise
version stacks the same operations in the framework's sequential container.
On Fashion-MNIST, after 30 epochs, the hidden layer of 256 units removes
about a fifth of the errors of softmax regression, with 26 times the
parameters and multiply-adds and, on a GPU, about the same training time per
epoch. No single class gains reliably. The four tops still hold most of the
errors, and the hidden units with the largest weights respond to parts of
garments, not to whole ones.

## Exercises

1. [code] **A hidden layer without activation.** Replace `relu` by the
   identity in `MLPScratch` and train the resulting model as in this
   section.
    1. Compare its validation accuracy after 30 epochs with that of the MLP
       and of softmax regression.
    1. After training, compute $\mathbf{W} = \mathbf{W}^{(1)}\mathbf{W}^{(2)}$
       and $\mathbf{b} = \mathbf{W}^{(2)\top}\mathbf{b}^{(1)} + \mathbf{b}^{(2)}$,
       and verify on a minibatch that $\mathbf{X}\mathbf{W} + \mathbf{b}$
       reproduces the logits of the model.
    1. Exercise 1 showed that such a network
       computes exactly the affine maps whose weight matrix has rank at most
       the hidden width. Explain why this restriction costs nothing here,
       and state below which hidden width it would reduce what the model
       can represent on Fashion-MNIST.
1. [code] **Recall across runs.** The recall table of this section holds
   one run of softmax regression and two runs of the MLP, too few to say
   which differences between the models are real.
    1. Train each model five times and report, for every class, the mean and
       the range of its recall. For which classes is the difference between
       the two models larger than the spread within either of them?
    1. Compute the column-normalized confusion matrix $\mathbf{C}$ of
       that section for one run of each model. Which
       confusions among the four tops does the hidden layer reduce, and
       which does it only move from one pair of classes to another?
    1. Keep a copy of the parameters of the MLP after each of its last five
       epochs, and classify each validation image by the average of the five
       predicted probabilities. Does the recall of the tops vary less
       between runs than that of a single epoch?
1. [code] **Single hidden unit.** Set `num_hiddens=1` in `MLPScratch`, so
   that its hidden layer has a single ReLU unit with weights
   $\mathbf{w}^{(1)} \in \mathbb{R}^{784 \times 1}$ and bias
   $b^{(1)} \in \mathbb{R}$. Each input $\mathbf{x} \in \mathbb{R}^{1 \times 784}$
   is then reduced to the scalar
   $t = \operatorname{ReLU}(\mathbf{x}\mathbf{w}^{(1)} + b^{(1)}) \geq 0$,
   and the logits are $\mathbf{o} = t\, \mathbf{w}^{(2)} + \mathbf{b}^{(2)}$
   with $\mathbf{w}^{(2)} \in \mathbb{R}^{1 \times 10}$ and
   $\mathbf{b}^{(2)} \in \mathbb{R}^{10}$.
    1. Show that each class is predicted on at most one interval of $t$, and
       that the order of these intervals along the $t$ axis is fixed by the
       entries of $\mathbf{w}^{(2)}$. Describe the regions of input space
       that the model assigns to the classes, and state what they imply for
       separating the ten Fashion-MNIST classes.
    1. Suppose that after some update
       $\mathbf{x}\mathbf{w}^{(1)} + b^{(1)} < 0$ for every training
       example. Which parameters receive nonzero gradients from then on, and
       what does the network predict? Why does the same event do little
       harm when it hits one of the 256 hidden units of this section?
    1. Train the model five times, and report for each run the validation
       accuracy and the fraction of validation examples on which $t > 0$,
       which `model.hidden` gives you. Compare any run in which this
       fraction is zero with your answer to (b).
1. [code] **Width.** Train `MLP` as in this section, for 30 epochs at
   learning rate 0.1, with hidden widths
   $h \in \{16, 64, 256, 1024, 4096\}$, and record the final validation
   accuracy and the time per epoch.
    1. Plot both against $h$ on a logarithmic axis. Beyond which width does
       the accuracy stop improving by more than its variation from epoch to
       epoch?
    1. The multiply-adds grow in proportion to $h$. Time one pass over the
       training data that only reads the minibatches, without any model,
       and use it to explain the part of the time curve that does not grow
       with $h$. From which width on does the arithmetic dominate on your
       hardware?
    1. Repeat the training at widths 64 and 1024 with learning rates
       $\{0.03, 0.1, 0.3\}$. Does the best learning rate depend on the
       width?
1. [code] **Initialization scale and depth.** ● `MLPScratch` draws its
   weights with standard deviation 0.01, which its argument `std` sets.
    1. Train the one-hidden-layer model with standard deviations 0.001,
       0.01, and 0.1, and plot the training and validation curves for each.
       How much do the curves differ, and does the difference persist as
       training continues? Relate your observation to the scale of the
       hidden activations and the logits on the first forward pass.
    1. Extend `MLPScratch` to two and to four hidden layers, each of
       width 256. Before running anything, predict which depth is more
       sensitive to the standard deviation, and why.
    1. Sweep the standard deviation over
       $\{0.001, 0.003, 0.01, 0.03, 0.1\}$ for both depths. For each depth,
       report the range for which the network reaches at least 70%
       validation accuracy within five epochs.
       that section explains the pattern and trains a
       network with ten hidden layers.

    *Adapted from Stanford CS231n,
    [Assignment 2](https://cs231n.github.io/assignments2023/assignment2/),
    FullyConnectedNets Inline Question 1.*

[Discussions](https://d2l.discourse.group/t/17985)